# Mapping VSIC VN to MCC on Google Colab

Notebook này hướng dẫn cách chạy pipeline mapping mã ngành VSIC Việt Nam sang mã MCC (Merchant Category Code) sử dụng GPU trên Google Colab và Ollama.

## 1. Kết nối Google Drive
Kết quả mapping và checkpoint sẽ được lưu trực tiếp lên Google Drive để tránh mất dữ liệu khi runtime bị reset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Cài đặt Ollama
Chúng ta sẽ cài đặt Ollama để chạy các mô hình LLM local trên GPU của Colab.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

## 3. Khởi động Ollama Server
Chạy Ollama server trong background.

In [ ]:
import os
import threading
import time

def run_ollama():
    os.system("ollama serve")

threading.Thread(target=run_ollama, daemon=True).start()
time.sleep(5) # Đợi server khởi động

## 4. Tải Mô hình (Models)
Tải mô hình Qwen 3.5 9B cho việc re-ranking và BGE-M3 cho embedding.

In [ ]:
!ollama pull qwen3.5:9b
!ollama pull bge-m3

## 5. Clone code và cài đặt Dependencies
Clone repository từ GitHub về Google Drive, sau đó cài các thư viện **tối thiểu** cho pipeline mapping.

> **Lưu ý:** Dùng `colab/requirements-mapping.txt` thay vì `requirements.txt` đầy đủ. File gốc chứa `surya-ocr` và dev tools, dễ gây xung đột dependency và bắt buộc restart runtime trên Colab.

In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/projects/mcc-lens"
REPO_URL = "https://github.com/hatuan314/mcc-lens.git"

os.makedirs(os.path.dirname(PROJECT_DIR), exist_ok=True)

if os.path.isdir(os.path.join(PROJECT_DIR, ".git")):
    os.chdir(PROJECT_DIR)
    !git pull
else:
    !git clone {REPO_URL} {PROJECT_DIR}
    os.chdir(PROJECT_DIR)

# Ensure Surya OCR (only needed for image-to-JSON feature) is NOT installed
# in this Colab runtime, to avoid httpx version conflicts.
!pip uninstall -y surya-ocr || true

!pip install -q -r colab/requirements-mapping.txt

## 6. Chạy Pipeline Mapping
Sử dụng flag `--gdrive-output-dir` để lưu kết quả vào thư mục trên Drive.

In [ ]:
import os

# Path relative to project root (works on both Colab and local)
project_root = os.getcwd()  # Should be PROJECT_DIR after cell 10
debug_log = os.path.join(project_root, ".cursor", "debug-c603c2.log")

print(f"Looking for log at: {debug_log}")

if os.path.exists(debug_log):
    with open(debug_log) as f:
        content = f.read()
    print(f"Log file size: {len(content)} bytes")
    print("=== LOG CONTENTS ===")
    print(content)
else:
    print(f"Log file NOT found.")
    cursor_dir = os.path.join(project_root, ".cursor")
    os.makedirs(cursor_dir, exist_ok=True)
    print(f"Created .cursor dir at: {cursor_dir}")
    print(f"cwd: {project_root}")
    print(f"Project dir listing: {os.listdir(project_root)}")

In [ ]:
!python3 main.py map-vsic-mcc \
  --vsic-input output/vsic-vn.json \
  --mcc-input output/mcc-visa.json \
  --gdrive-output-dir /content/drive/MyDrive/projects/mcc-lens \
  --llm-model qwen3.5:9b \
  --resume